nabu ROM build on the G4 runtime.

In [ ]:
#@title Pins and paths
import os
os.environ.update(
    BASE_MANIFEST_COMMIT="eabe68377217a88c81fa933db0136ea4146ff369",
    REPO_TOOL_REVISION="v2.69",
    DEVICE_TREE_COMMIT="c5bbb2a541226967a69c829142f0e6fb53187717",
    VENDOR_BLOBS_COMMIT="b58f316e3c051721dda4af44277e6d47ac43bcfb",
    HARDWARE_XIAOMI_COMMIT="af5766acc6d6766eddeb52574996247128428ff6",
    GAPPS_COMMIT="39bd3a09640efb81234cdbd8ab98ab71541d5d46",
    KERNEL_COMMIT="2b1fcdf920b8c8bc8cec5b1c632f5e9ae6d25968",
    KERNEL_CLANG_VERSION="clang-r563880c",
    MKDTBOIMG_SHA256="ea9507427f4f691e4de27ba6918db7880511448e9dd63a80874a79c1fcaffd82",
    SOURCE_TREE="/content/rom/lineage",
    LOG_DIRECTORY="/content/rom/logs",
    OUT_DISK_DIRECTORY="/content/rom/out-disk",
    KERNEL_RAM_DISK="/content/rom/kernel-ram",
)

In [ ]:
%%shell
set -e
RAM_GIB=$(awk '/^MemTotal:/ {printf "%d", $2 / 1048576}' /proc/meminfo)
echo "runtime: $(nproc) CPU, $RAM_GIB GB, up $(awk '{printf "%d", $1 / 60}' /proc/uptime) min"
[ "$RAM_GIB" -ge 160 ] || { echo "RAM under 160 GB: switch to G4" >&2; exit 1; }
df --block-size=1G --output=size,used,avail /content | tail --lines=1 | awk '{print "disk: " $1 " GB, " $2 " used, " $3 " free"}'
mkdir --parents "${SOURCE_TREE:?}" "$LOG_DIRECTORY" "$OUT_DISK_DIRECTORY"
if ! mountpoint --quiet "$SOURCE_TREE"; then
    mount --types tmpfs --options size=140G tmpfs "$SOURCE_TREE"
fi
mkdir --parents "$SOURCE_TREE/out"
if ! mountpoint --quiet "$SOURCE_TREE/out"; then
    mount --bind "$OUT_DISK_DIRECTORY" "$SOURCE_TREE/out"
fi
apt-get update > "$LOG_DIRECTORY/packages.log" 2>&1 || { echo "apt-get update failed: rerun this cell" >&2; tail --lines=5 "$LOG_DIRECTORY/packages.log" >&2; exit 1; }
apt-get install --yes bc bison build-essential curl erofs-utils flex g++-multilib gcc-multilib git git-lfs gnupg gperf imagemagick protobuf-compiler python3-protobuf lib32readline-dev lib32z1-dev libdw-dev libelf-dev libgnutls28-dev lz4 libsdl1.2-dev libssl-dev libxml2 libxml2-utils lzop pngcrush rsync schedtool squashfs-tools xsltproc xxd zip zlib1g-dev python-is-python3 repo cpio time >> "$LOG_DIRECTORY/packages.log" 2>&1 || { echo "package install failed: rerun this cell" >&2; tail --lines=5 "$LOG_DIRECTORY/packages.log" >&2; exit 1; }

In [ ]:
%%shell
set -e
SYNC_LOG=${LOG_DIRECTORY:?}/sync.log
if [ -e "$SOURCE_TREE/.repo/nabu-pruned" ]; then
    echo "tree pruned: start a fresh runtime" >&2
    exit 1
fi
SYNC_START=$(date +%s)
cd "$SOURCE_TREE"
repo init --manifest-url=https://github.com/LineageOS/android.git --manifest-branch="$BASE_MANIFEST_COMMIT" --repo-rev="$REPO_TOOL_REVISION" --depth=1 --git-lfs --no-clone-bundle > "$SYNC_LOG" 2>&1 || { echo "repo init failed: rerun this cell" >&2; tail --lines=20 "$SYNC_LOG" >&2; exit 1; }
mkdir --parents .repo/local_manifests
cat > .repo/local_manifests/nabu.xml <<MANIFEST
<?xml version="1.0" encoding="UTF-8"?>
<manifest>
  <remote name="nabu-github" fetch="https://github.com/" />
  <project path="device/xiaomi/nabu" name="djo-n/android_device_xiaomi_nabu" remote="nabu-github" revision="$DEVICE_TREE_COMMIT" upstream="refs/heads/nabu" />
  <project path="vendor/xiaomi/nabu" name="djo-n/proprietary_vendor_xiaomi_nabu" remote="nabu-github" revision="$VENDOR_BLOBS_COMMIT" upstream="refs/heads/nabu" />
  <project path="kernel/xiaomi/nabu" name="djo-n/android_kernel_xiaomi_nabu" remote="nabu-github" revision="$KERNEL_COMMIT" upstream="refs/heads/nabu" />
  <project path="hardware/xiaomi" name="LineageOS/android_hardware_xiaomi" remote="nabu-github" revision="$HARDWARE_XIAOMI_COMMIT" upstream="refs/heads/lineage-23.2" />
  <project path="vendor/gapps" name="MindTheGapps/vendor_gapps" remote="nabu-github" revision="$GAPPS_COMMIT" upstream="refs/heads/baklava" />
</manifest>
MANIFEST
repo sync --current-branch --no-tags --retry-fetches=8 >> "$SYNC_LOG" 2>&1 || { echo "sync failed: rerun this cell, it resumes" >&2; tail --lines=20 "$SYNC_LOG" >&2; exit 1; }
echo "sync: $(( ($(date +%s) - SYNC_START) / 60 )) min"

In [ ]:
%%shell
set -e
cd "${SOURCE_TREE:?}"
if [ -e .repo/nabu-patched ]; then
    echo "patches: already applied"
    exit 0
fi
PATCH_ROOT=device/xiaomi/nabu/patches
if [ ! -d "$PATCH_ROOT" ]; then
    echo "no $PATCH_ROOT: this cell leaves together with the last patch" >&2
    exit 1
fi
while IFS= read -r -d '' PATCH_PATH; do
    PROJECT_PATH=$(dirname "${PATCH_PATH#"$PATCH_ROOT"/}")
    git -C "$PROJECT_PATH" apply "$SOURCE_TREE/$PATCH_PATH" || { echo "patch does not apply: $PATCH_PATH" >&2; exit 1; }
    echo "patch: $PATCH_PATH"
done < <(find "$PATCH_ROOT" -name '*.patch' -print0 | sort --zero-terminated)
touch .repo/nabu-patched

In [ ]:
%%shell
set -e
set -o pipefail
KERNEL_GIT_DIRECTORY=${SOURCE_TREE:?}/.repo/project-objects/djo-n/android_kernel_xiaomi_nabu.git
[ -d "$KERNEL_GIT_DIRECTORY" ] || { echo "no kernel objects at $KERNEL_GIT_DIRECTORY" >&2; exit 1; }
touch "$SOURCE_TREE/.repo/nabu-pruned"
while IFS= read -r -d '' GIT_DIRECTORY; do
    if [ "$GIT_DIRECTORY" = "$KERNEL_GIT_DIRECTORY" ]; then
        continue
    fi
    git --git-dir="$GIT_DIRECTORY" cat-file --batch-all-objects --batch-check='%(objectname) %(objecttype)' | awk '$2 == "commit" || $2 == "tag" {print $1}' > /tmp/keep-objects
    [ -s /tmp/keep-objects ] || { echo "no commits in $GIT_DIRECTORY" >&2; exit 1; }
    git --git-dir="$GIT_DIRECTORY" pack-objects --quiet "$GIT_DIRECTORY/objects/pack/commits-only" < /tmp/keep-objects > /dev/null
    find "$GIT_DIRECTORY/objects/pack" -type f ! -name 'commits-only-*' -delete
    find "$GIT_DIRECTORY/objects" -mindepth 2 -maxdepth 2 -type f -path "$GIT_DIRECTORY/objects/??/*" -delete
done < <(find "$SOURCE_TREE/.repo/project-objects" -type d -name '*.git' -print0 | sort --zero-terminated)
cd "$SOURCE_TREE"
repo manifest --revision-as-HEAD --output-file="${LOG_DIRECTORY:?}/pruned-manifest.xml"
echo ".repo: $(du --summarize --block-size=1G .repo | cut --fields=1) GB"
df --block-size=1G --output=used,avail "$SOURCE_TREE" | tail --lines=1 | awk '{print "RAM disk: " $1 " GB used, " $2 " free"}'

In [ ]:
%%shell
set -e
set -o pipefail
KERNEL_SOURCE=${SOURCE_TREE:?}/kernel/xiaomi/nabu
KERNEL_CONFIG_DIRECTORY=$KERNEL_SOURCE/arch/arm64/configs/vendor
KERNEL_OUT=$KERNEL_RAM_DISK/out
KERNEL_LOG=$LOG_DIRECTORY/kernel.log
PREBUILT_KERNEL_DIRECTORY=$SOURCE_TREE/prebuilts/nabu-kernel
CLANG_DIRECTORY=$SOURCE_TREE/prebuilts/clang/host/linux-x86/$KERNEL_CLANG_VERSION
MKDTBOIMG_PATH=$SOURCE_TREE/system/libufdt/utils/src/mkdtboimg.py
rm --recursive --force "$PREBUILT_KERNEL_DIRECTORY"
[ -x "$CLANG_DIRECTORY/bin/clang" ] || { echo "missing $CLANG_DIRECTORY/bin/clang" >&2; exit 1; }
echo "$MKDTBOIMG_SHA256  $MKDTBOIMG_PATH" | sha256sum --check --quiet
if mountpoint --quiet "$KERNEL_RAM_DISK"; then
    umount "$KERNEL_RAM_DISK"
fi
mkdir --parents "$KERNEL_RAM_DISK"
mount --types tmpfs --options size=20G tmpfs "$KERNEL_RAM_DISK"
mkdir --parents "$KERNEL_OUT"
KERNEL_START=$(date +%s)
export PATH="$CLANG_DIRECTORY/bin:$PATH"
export KBUILD_BUILD_USER=nabu-rom
export KBUILD_BUILD_HOST=builder
KBUILD_BUILD_TIMESTAMP=$(TZ=UTC git -C "$KERNEL_SOURCE" log --max-count=1 --format=%cd --date=format-local:'%a %b %d %H:%M:%S UTC %Y')
[ -n "$KBUILD_BUILD_TIMESTAMP" ] || { echo "no committer date in $KERNEL_SOURCE" >&2; exit 1; }
export KBUILD_BUILD_TIMESTAMP
MAKE_FLAGS=(O="$KERNEL_OUT" ARCH=arm64 CROSS_COMPILE=aarch64-linux-gnu- CC=clang LD="$CLANG_DIRECTORY/bin/ld.lld" AR="$CLANG_DIRECTORY/bin/llvm-ar" LLVM=1 LLVM_IAS=1 KCPPFLAGS="-fmacro-prefix-map=$KERNEL_SOURCE/=" HOSTCC="$CLANG_DIRECTORY/bin/clang" HOSTCXX="$CLANG_DIRECTORY/bin/clang++" HOSTLD="$CLANG_DIRECTORY/bin/ld.lld" HOSTAR="$CLANG_DIRECTORY/bin/llvm-ar")
cd "$KERNEL_SOURCE"
cp "$KERNEL_CONFIG_DIRECTORY/sm8150-perf_defconfig" "$KERNEL_OUT/.config"
make "${MAKE_FLAGS[@]}" olddefconfig > "$KERNEL_LOG" 2>&1 || { echo "config failed: sm8150-perf_defconfig" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
scripts/kconfig/merge_config.sh -m -O "$KERNEL_OUT" "$KERNEL_OUT/.config" "$KERNEL_CONFIG_DIRECTORY/debugfs.config" >> "$KERNEL_LOG" 2>&1 || { echo "config failed: debugfs.config" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
make "${MAKE_FLAGS[@]}" olddefconfig >> "$KERNEL_LOG" 2>&1 || { echo "config failed: debugfs.config" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
scripts/kconfig/merge_config.sh -m -O "$KERNEL_OUT" "$KERNEL_OUT/.config" "$KERNEL_CONFIG_DIRECTORY/xiaomi/sm8150-common.config" >> "$KERNEL_LOG" 2>&1 || { echo "config failed: sm8150-common.config" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
make "${MAKE_FLAGS[@]}" olddefconfig >> "$KERNEL_LOG" 2>&1 || { echo "config failed: sm8150-common.config" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
scripts/kconfig/merge_config.sh -m -O "$KERNEL_OUT" "$KERNEL_OUT/.config" "$KERNEL_CONFIG_DIRECTORY/xiaomi/nabu.config" >> "$KERNEL_LOG" 2>&1 || { echo "config failed: nabu.config" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
make "${MAKE_FLAGS[@]}" olddefconfig >> "$KERNEL_LOG" 2>&1 || { echo "config failed: nabu.config" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
grep --quiet --fixed-strings --line-regexp 'CONFIG_HID_BATTERY_STRENGTH=y' "$KERNEL_OUT/.config" || { echo "missing CONFIG_HID_BATTERY_STRENGTH=y" >&2; exit 1; }
if grep --extended-regexp '^CONFIG_KSU[A-Z_]*=[ym]$' "$KERNEL_OUT/.config"; then
    echo "KSU enabled" >&2
    exit 1
fi
set +e
/usr/bin/time --verbose --output="$KERNEL_RAM_DISK/time.txt" make --jobs="$(nproc)" "${MAKE_FLAGS[@]}" Image.gz dtbs >> "$KERNEL_LOG" 2>&1
KERNEL_STATUS=$?
set -e
PEAK_GIB=$(awk -F': ' '/Maximum resident set size/ {printf "%.1f", $2 / 1048576}' "$KERNEL_RAM_DISK/time.txt")
echo "kernel ${KERNEL_COMMIT:0:12}: make $KERNEL_STATUS, $(( ($(date +%s) - KERNEL_START) / 60 )) min, peak $PEAK_GIB GB, $(grep --extended-regexp '^CONFIG_(LTO_CLANG|THINLTO|LTO_NONE)=y' "$KERNEL_OUT/.config" | paste --serial --delimiters=' ')"
if [ "$KERNEL_STATUS" -ne 0 ]; then
    echo "error: $(grep --max-count=1 --extended-regexp 'error:|Error [0-9]+|Killed' "$KERNEL_LOG")"
    tail --lines=30 "$KERNEL_LOG"
    exit "$KERNEL_STATUS"
fi
make "${MAKE_FLAGS[@]}" headers_install >> "$KERNEL_LOG" 2>&1 || { echo "headers_install failed" >&2; tail --lines=15 "$KERNEL_LOG" >&2; exit 1; }
cd "$KERNEL_OUT/arch/arm64/boot/dts"
find . -type f -name '*.dtb' | LC_ALL=C sort > "$KERNEL_RAM_DISK/dtb-order.txt"
find . -type f -name '*.dtbo' | LC_ALL=C sort > "$KERNEL_RAM_DISK/dtbo-order.txt"
[ -s "$KERNEL_RAM_DISK/dtb-order.txt" ] || { echo "no dtb files built" >&2; exit 1; }
[ -s "$KERNEL_RAM_DISK/dtbo-order.txt" ] || { echo "no dtbo files built" >&2; exit 1; }
mkdir --parents "$PREBUILT_KERNEL_DIRECTORY/dtb"
cp "$KERNEL_OUT/arch/arm64/boot/Image.gz" "$PREBUILT_KERNEL_DIRECTORY/Image.gz"
xargs --arg-file="$KERNEL_RAM_DISK/dtb-order.txt" cat > "$PREBUILT_KERNEL_DIRECTORY/dtb/nabu.dtb"
xargs --arg-file="$KERNEL_RAM_DISK/dtbo-order.txt" python3 -I "$MKDTBOIMG_PATH" create "$PREBUILT_KERNEL_DIRECTORY/dtbo.img" --page_size=4096
tar --create --gzip --file="$PREBUILT_KERNEL_DIRECTORY/kernel-headers.tar.gz" --directory="$KERNEL_OUT" usr
echo "release: $(cat "$KERNEL_OUT/include/config/kernel.release")"
echo "version: $(grep UTS_VERSION "$KERNEL_OUT/include/generated/compile.h" | cut --delimiter='"' --fields=2)"
echo "clang: $("$CLANG_DIRECTORY/bin/clang" --version | sed --quiet '1p')"
echo "dtbs: $(wc --lines < "$KERNEL_RAM_DISK/dtb-order.txt"), dtbos: $(wc --lines < "$KERNEL_RAM_DISK/dtbo-order.txt")"
cd "$PREBUILT_KERNEL_DIRECTORY"
sha256sum Image.gz dtb/nabu.dtb dtbo.img kernel-headers.tar.gz
umount "$KERNEL_RAM_DISK"

In [ ]:
%%shell
set -e
MEMORY_AVAILABLE_GIB=$(awk '/^MemAvailable:/ {printf "%d", $2 / 1048576}' /proc/meminfo)
HIGHMEM_JOBS=$(( (MEMORY_AVAILABLE_GIB - 32) / 8 ))
[ "$HIGHMEM_JOBS" -ge 2 ] || { echo "RAM free: $MEMORY_AVAILABLE_GIB GB, needs 48" >&2; exit 1; }
export NINJA_HIGHMEM_NUM_JOBS=$HIGHMEM_JOBS
echo "RAM free: $MEMORY_AVAILABLE_GIB GB, highmem jobs: $HIGHMEM_JOBS"
BUILD_LOG=${LOG_DIRECTORY:?}/build.log
DISK_LOG=$LOG_DIRECTORY/disk.log
rm --force "$DISK_LOG"
(
    MINUTE=0
    while true; do
        USED_MEGABYTES=$(df --block-size=1M --output=used /content | tail --lines=1 | tr --delete ' ')
        PHYSICAL_FREE_MEGABYTES=$(df --block-size=1M --output=avail /kaggle/input | tail --lines=1 | tr --delete ' ')
        MEMORY_AVAILABLE_MEGABYTES=$(awk '/^MemAvailable:/ {printf "%d", $2 / 1024}' /proc/meminfo)
        echo "$(date +%s) $USED_MEGABYTES $PHYSICAL_FREE_MEGABYTES $MEMORY_AVAILABLE_MEGABYTES" >> "$DISK_LOG"
        MINUTE=$((MINUTE + 1))
        if [ $((MINUTE % 15)) -eq 0 ]; then
            printf '%d:%02d disk %d GB, physical free %d GB, RAM free %d GB, %s\n' $((MINUTE / 60)) $((MINUTE % 60)) $((USED_MEGABYTES / 1024)) $((PHYSICAL_FREE_MEGABYTES / 1024)) $((MEMORY_AVAILABLE_MEGABYTES / 1024)) "$(tail --bytes=2000 "$BUILD_LOG" | tr '\r' '\n' | grep --extended-regexp '^\[ *[0-9]+% ' | tail --lines=1 | cut --characters=1-60)"
        fi
        sleep 60
    done
) < /dev/null &
MONITOR_PID=$!
BUILD_START=$(date +%s)
set +e
(
    export BUILD_USERNAME=nabu
    export BUILD_HOSTNAME=builder
    export TARGET_PREBUILT_KERNEL_HEADERS=prebuilts/nabu-kernel/kernel-headers.tar.gz
    cd "$SOURCE_TREE"
    mkdir --parents vendor/lineage-priv/keys
    echo 'PRODUCT_EXTRA_RECOVERY_KEYS :=' > vendor/lineage-priv/keys/keys.mk
    source build/envsetup.sh || exit 1
    breakfast nabu userdebug || exit 1
    m bacon target-files-package otatools-package
) < /dev/null > "$BUILD_LOG" 2>&1
BUILD_STATUS=$?
kill "$MONITOR_PID" || echo "monitor stopped early"
echo "build: exit $BUILD_STATUS, $(( ($(date +%s) - BUILD_START) / 60 )) min, out $(du --summarize --block-size=1G "$SOURCE_TREE/out" | cut --fields=1) GB"
awk '
    NR == 1 {
        peak = $2
        physical = $3
        memory = $4
    }
    $2 > peak {
        peak = $2
    }
    $3 < physical {
        physical = $3
    }
    $4 < memory {
        memory = $4
    }
    END {
        printf "peak disk %d GB, min physical free %d GB, min RAM free %d GB\n", peak / 1024, physical / 1024, memory / 1024
    }
' "$DISK_LOG"
if [ "$BUILD_STATUS" -ne 0 ]; then
    echo "last step: $(tail --bytes=3000 "$BUILD_LOG" | tr '\r' '\n' | grep --extended-regexp '^\[ *[0-9]+% ' | tail --lines=1 | cut --characters=1-100)"
    du --one-file-system --max-depth=3 --block-size=1G "$SOURCE_TREE/out" 2> /dev/null | sort --numeric-sort --reverse | sed --quiet "1,25s|$SOURCE_TREE/||p"
    if [ -s "$SOURCE_TREE/out/error.log" ]; then
        echo "out/error.log:"
        head --lines=80 "$SOURCE_TREE/out/error.log"
    fi
    echo "build.log:"
    tail --lines=30 "$BUILD_LOG"
    exit "$BUILD_STATUS"
fi

In [ ]:
%%shell
set -e
OUTPUT_DIRECTORY=${SOURCE_TREE:?}/out/target/product/nabu
TARGET_FILES_DIRECTORY=$OUTPUT_DIRECTORY/obj/PACKAGING/target_files_intermediates
OTATOOLS=$(find "$SOURCE_TREE/out/soong/.intermediates/build/make/tools/otatools_package/otatools-package" -name otatools.zip)
[ -f "$OTATOOLS" ] || { echo "otatools.zip: not exactly one" >&2; exit 1; }
cd "$OUTPUT_DIRECTORY"
ROM_ZIP=$(ls --sort=time lineage-*-UNOFFICIAL-nabu.zip | sed --quiet '1p')
sha256sum "$ROM_ZIP" boot.img vendor_boot.img dtbo.img vbmeta.img > nabu-rom-SHA256SUMS
(cd "$TARGET_FILES_DIRECTORY" && sha256sum lineage_nabu-target_files.zip) > nabu-rom-signing-SHA256SUMS
(cd "$(dirname "$OTATOOLS")" && sha256sum otatools.zip) >> nabu-rom-signing-SHA256SUMS
tar --create --file /content/nabu-rom-out.tar "$ROM_ZIP" boot.img vendor_boot.img dtbo.img vbmeta.img nabu-rom-SHA256SUMS nabu-rom-signing-SHA256SUMS --directory "$TARGET_FILES_DIRECTORY" lineage_nabu-target_files.zip --directory "$(dirname "$OTATOOLS")" otatools.zip
echo "download: /content/nabu-rom-out.tar"